# Лекция 09 · Цели проекта и Human-level performance / Project goals and human-level performance

**RU.** ДЗ №9 · MET-09 · 100 баллов. Четыре функции: матрица ошибок, метрики, выбор порога, анализ по исходным классам. Выполняйте блоки по порядку на CPU. Положительный класс: dog ИЛИ frog, отрицательный: cat. Это учебная классификация животных, НЕ NSFW-детектор. Не интерпретируйте frog как вредный контент.

**EN.** Homework09 · MET-09 · 100 points. Four functions: confusion counts, metrics, threshold selection, original-class error analysis. Run in order on CPU. Positive means dog OR frog; negative means cat. This is an animal-classification exercise, NOT an NSFW detector. Frog is not a harmful-content label.

**Данные / Data.** [CIFAR-10, Krizhevsky, Nair & Hinton](https://www.cs.toronto.edu/~kriz/cifar.html): real 32×32 RGB photographs. One observation is an image; 192 features are 4×4 spatial block means per channel; target is cat=0, dog/frog=1. The first 1000 images per class from official training batches are split 2250/750, stratified seed909; the first 200/class from official test give 600 held-out images. Original source IDs are retained. Official train and test are never pooled. This small fixed subset is not the full CIFAR benchmark; no inference about NSFW or human performance is supported. Dataset images are for teaching; retain source attribution and consult source terms before redistribution.

**RU.** Модель фиксирована до подбора порога: StandardScaler и LogisticRegression(C=0.1), fit только на train. Среднее по пространственным блокам — детерминированное преобразование каждого изображения. Test открываем после фиксации двух правил выбора. Сценарий цены FN=5, FP=1 — учебное допущение, не измеренные затраты. Человеческие оценки здесь не собирались.

**EN.** Fix StandardScaler and LogisticRegression(C=0.1) before threshold selection; fit on train only. Spatial block averaging is a deterministic per-image transformation. Open test after freezing both selection rules. FN cost5 and FP cost1 are teaching assumptions, not measured costs. No human annotations were collected here.

Dependencies: NumPy, scikit-learn, matplotlib; available in Colab. First run downloads the official 163MiB archive and verifies its checksum. Local cache: build/data/cifar-10-python.tar.gz or ./cifar-10-python.tar.gz.

## Сдача / Submission
Заполните имя и группу, реализуйте четыре функции, выполните проверки и сдайте ipynb в папку курса. Не редактируйте проверочную ячейку. Шкала:20+20+25+20+15=100. Результат тестов — исходный балл; сроки и практика учитываются внешним проверяющим. / Fill identity, implement four functions, run checks and submit the notebook. Do not edit tests. Tests report raw points; deadline and practice penalties are applied externally.

Срок — [матрица по группам](https://docs.google.com/spreadsheets/d/15N5GGHujXPAzRNuyOfmooT7A7sKhXgrMfWHkUkvYYiM/edit), ДЗ09. Просрочка: линейный штраф за14дней; отсутствие практики09:×0.9 один раз после просрочки. / Group deadline matrix, homework09; linear14-day late penalty, then missing practice09×0.9 once.

Дополнительно представьте подробный отчёт RU или EN: два validation-порога и правило ничьей; test-метрики и baseline; стоимость FN/FP; срезы cat/dog/frog; четыре реальные ошибочные фотографии; ограничения и протокол возможной человеческой оценки. Эти пояснения рассматривает преподаватель; автоматические100баллов относятся к функциям. / Also include a report in RU or EN covering frozen thresholds, tie rule, test metrics, baseline, costs, class slices, four real error images, limitations and a proposed human-evaluation protocol. The instructor reviews the report; automatic100points cover functions.

In [ ]:
student_first = ""
student_last = ""
student_group = ""
assignment_id = "MET-09"

In [ ]:
import json, pickle, tarfile, hashlib, urllib.request  # RU: читаем проверенный архив и формируем отчёт; EN: read a verified archive and produce a report.
from pathlib import Path  # RU: пути к локальному кэшу; EN: local cache paths.
import numpy as np  # RU: операции с изображениями и метками; EN: image and label operations.
import matplotlib.pyplot as plt  # RU: показываем реальные фотографии; EN: display real photographs.
from sklearn.model_selection import train_test_split  # RU: стратифицированное разбиение; EN: stratified splitting.
from sklearn.pipeline import make_pipeline  # RU: связываем масштабирование и модель; EN: combine scaling and model.
from sklearn.preprocessing import StandardScaler  # RU: оцениваем масштабы на train; EN: estimate scales on training data.
from sklearn.linear_model import LogisticRegression  # RU: фиксированный линейный baseline; EN: fixed linear baseline.
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score  # RU: независимые проверки; EN: independent test references.
archive = next((p for p in [Path('build/data/cifar-10-python.tar.gz'), Path('../build/data/cifar-10-python.tar.gz'), Path('data/cifar-10-python.tar.gz'), Path('cifar-10-python.tar.gz')] if p.exists()), Path('cifar-10-python.tar.gz'))  # RU: используем существующий кэш; EN: reuse an existing cache.
if not archive.exists():  # RU: загружаем только при отсутствии; EN: download only when missing.
    urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', archive)  # RU: официальный источник CIFAR; EN: official CIFAR source.
assert hashlib.md5(archive.read_bytes()).hexdigest() == 'c58f30108f718f92721af3b95e74349a'  # RU: проверяем целостность до pickle; EN: check integrity before unpickling.
with tarfile.open(archive) as tar:  # RU: читаем конкретные файлы без распаковки путей; EN: read named members without extracting paths.
    batches = [pickle.load(tar.extractfile(f'cifar-10-batches-py/data_batch_{i}'), encoding='bytes') for i in range(1, 6)]  # RU: официальная обучающая часть; EN: official training batches.
    held = pickle.load(tar.extractfile('cifar-10-batches-py/test_batch'), encoding='bytes')  # RU: отдельная тестовая часть; EN: separate official test batch.
raw_x = np.concatenate([b[b'data'] for b in batches])  # RU: объединяем только train-пакеты; EN: concatenate training batches only.
raw_y = np.concatenate([b[b'labels'] for b in batches])  # RU: сохраняем исходные классы; EN: preserve original classes.
source_ids = np.concatenate([np.flatnonzero(raw_y == k)[:1000] for k in [3, 5, 6]])  # RU: фиксируем подмножество до обучения; EN: fix the subset before fitting.
images = raw_x[source_ids].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)  # RU: восстанавливаем RGB; EN: restore RGB layout.
original_class = raw_y[source_ids]  # RU: классы для анализа срезов; EN: labels for slice analysis.
y = (original_class != 3).astype(int)  # RU: cat=0, dog/frog=1; EN: cat=0, dog/frog=1.
train, valid = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y, random_state=909)  # RU: разделяем обучение и выбор порога; EN: separate fitting and threshold selection.
def features(img):  # RU: детерминированные признаки изображения; EN: deterministic image features.
    return img.astype(float).reshape(-1, 8, 4, 8, 4, 3).mean(axis=(2, 4)).reshape(-1, 192) / 255  # RU: усредняем блоки 4×4 отдельно по RGB; EN: average 4×4 blocks separately per RGB channel.
x = features(images)  # RU: преобразуем train-источник без оценки статистик; EN: transform the training source without fitted statistics.
model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000, random_state=909))  # RU: заранее фиксируем модель; EN: predeclare the model.
model.fit(x[train], y[train])  # RU: fit только обучающей части; EN: fit training data only.
val_y = y[valid]  # RU: метки для выбора порога; EN: labels for threshold selection.
val_scores = model.predict_proba(x[valid])[:, 1]  # RU: score положительного класса; EN: positive-class scores.
assert not set(train) & set(valid)  # RU: проверяем непересечение индексов; EN: verify disjoint indices.
print('train/validation:', len(train), len(valid))  # RU: ожидаем 2250/750; EN: expect 2250/750.

## 1. Матрица ошибок / Confusion counts
**RU.** Реализуйте `confusion_counts(y_true,y_pred)` → dict с целыми tn,fp,fn,tp. Оба входа — непустые одномерные бинарные массивы одинаковой длины; при неверной форме/метках ValueError. Не изменяйте входы. Сначала нарисуйте на бумаге таблицу «истина × прогноз»; затем посчитайте масками. Используйте прогноз `val_scores>=0.5` и объясните, какой реальный класс скрывается за FP.
**EN.** Implement the function returning integer tn,fp,fn,tp. Require aligned nonempty one-dimensional binary arrays; reject invalid shapes/labels with ValueError. Do not mutate inputs. Sketch truth versus prediction, then use Boolean masks. Evaluate `val_scores>=0.5` and explain which original class produces FP.

In [ ]:
def confusion_counts(y_true, y_pred):  # RU: считаем четыре исхода бинарного решения; EN: count four binary outcomes.
    raise NotImplementedError("Implement")

## 2. Метрики и baseline / Metrics and baseline
**RU.** Реализуйте `binary_metrics(counts)` → dict accuracy,precision,recall,f1. Вход — четыре неотрицательных целых счётчика. Любой нулевой знаменатель даёт 0.0. Сравните модель при0.5 с решением «всегда положительный класс». Почему F1=0.8 само по себе не доказывает пользу модели? Посчитайте FP и FN, а не только F1.
**EN.** Return accuracy,precision,recall,f1 from four nonnegative integer counts. Define a zero denominator as0.0. Compare threshold0.5 with always-positive predictions. Why does F1=0.8 alone fail to prove utility? Inspect FP and FN as well as F1.

In [ ]:
def binary_metrics(counts):  # RU: вычисляем метрики по счётчикам; EN: compute metrics from counts.
    raise NotImplementedError("Implement")

## 3. Выбор порога / Threshold selection
**RU.** Реализуйте `choose_threshold(y_true,scores,objective='f1',fn_cost=5,fp_cost=1)` → float. Scores — конечные числа[0,1], форма как у y. Кандидаты: уникальные значения `[0]+scores+[1]`; правило прогноза `>=`. Максимизируйте F1 либо минимизируйте `fn_cost*FN+fp_cost*FP`. При точном равенстве выбирайте наибольший порог. Неверные scores/критерий/отрицательная цена: ValueError. Выберите оба порога на validation и сохраните их ДО открытия test. Это два заранее заявленных сценария, нельзя выбирать между ними по test.
**EN.** Return a float threshold. Finite scores in[0,1] must align with y. Candidates are unique `[0]+scores+[1]`; predict using`>=`. Maximize F1 or minimize the declared FP/FN cost. Exact ties choose the largest threshold. Reject invalid scores/objectives/negative costs. Freeze both validation thresholds before opening test; do not choose between scenarios using test.

In [ ]:
def choose_threshold(y_true, scores, objective='f1', fn_cost=5.0, fp_cost=1.0):  # RU: выбираем порог исключительно по validation; EN: select a threshold using validation only.
    raise NotImplementedError("Implement")

## 4. Срезы и test / Slices and test
**RU.** Реализуйте `slice_errors(original_labels,y_true,y_pred)` → dict: ключ исходный класс, значение `n,errors,error_rate`. Входы согласованы, классы могут отсутствовать — возвращайте только присутствующие. После фиксации порогов из блока3 откройте test, оцените оба сценария и baseline, выведите срезы cat/dog/frog и четыре ошибочные фотографии. Не меняйте порог после просмотра. Письменно: какие ошибки дорого стоят, что скрывает общая F1, почему мы не измерили human-level performance?
**EN.** Return a dictionary keyed by each observed original class, with n,errors,error_rate. Inputs are aligned; omit absent classes. After freezing thresholds in block3, open test, evaluate both scenarios and baseline, report cat/dog/frog slices and four mistaken photographs. Do not retune after inspection. Explain expensive errors, what overall F1 hides, and why this experiment does not measure human-level performance.

In [ ]:
def slice_errors(original_labels, y_true, y_pred):  # RU: считаем ошибки отдельно по исходным классам; EN: count errors by original class.
    raise NotImplementedError("Implement")

## Автопроверка / Autograder

In [ ]:
checks = []
def check(name, points, function):
    try:
        function(); checks.append({'test':name, 'points':points, 'passed':True})
    except Exception as exc:
        checks.append({'test':name, 'points':0, 'passed':False, 'error':str(exc)[:180]})
def test_counts():
    for ids in [np.arange(len(val_y)), np.arange(1), np.arange(50, 100)]:
        for threshold in [0.0, 0.5, 1.0]:
            yy=val_y[ids]; pp=val_scores[ids]>=threshold; old=yy.copy()
            got=confusion_counts(yy,pp)
            assert got==dict(zip(['tn','fp','fn','tp'],confusion_matrix(yy,pp,labels=[0,1]).ravel()))
            np.testing.assert_array_equal(yy,old)
    for yy,pp in [(val_y[:0],val_y[:0]),(val_y[:3],val_y[:2]),(val_y[:3]+2,val_y[:3])]:
        try:confusion_counts(yy,pp)
        except ValueError:pass
        else:raise AssertionError('Must reject invalid labels/shapes')
def test_metrics():
    for threshold in [0.0,0.5,1.0]:
        pp=val_scores>=threshold; c=dict(zip(['tn','fp','fn','tp'],confusion_matrix(val_y,pp,labels=[0,1]).ravel())); got=binary_metrics(c)
        expected={'accuracy':float(np.mean(val_y==pp)),'precision':precision_score(val_y,pp,zero_division=0),'recall':recall_score(val_y,pp,zero_division=0),'f1':f1_score(val_y,pp,zero_division=0)}
        for k in expected:assert np.isclose(got[k],expected[k]), k
    assert all(v==0 for v in binary_metrics(dict(tn=0,fp=0,fn=0,tp=0)).values())
def test_f1():
    for ids in [np.arange(100),np.arange(100,200),np.arange(1)]:
        yy=val_y[ids];ss=val_scores[ids];candidates=np.unique(np.r_[0,ss,1]);expected=max(candidates,key=lambda t:(f1_score(yy,ss>=t,zero_division=0),t))
        assert choose_threshold(yy,ss)==expected
    for ss in [val_scores[:3]+2,np.array([np.nan]*3)]:
        try:choose_threshold(val_y[:3],ss)
        except ValueError:pass
        else:raise AssertionError('Invalid scores accepted')
def test_cost():
    yy=val_y[:100];ss=val_scores[:100];candidates=np.unique(np.r_[0,ss,1])
    for fn,fp in [(5,1),(1,5),(0,0)]:
        expected=min(candidates,key=lambda t:(fn*np.sum((yy==1)&(ss<t))+fp*np.sum((yy==0)&(ss>=t)),-t))
        assert choose_threshold(yy,ss,'cost',fn,fp)==expected
    for objective,fn,fp in [('unknown',1,1),('cost',-1,1)]:
        try:choose_threshold(yy,ss,objective,fn,fp)
        except ValueError:pass
        else:raise AssertionError('Invalid objective/cost accepted')
def test_slices():
    for ids in [np.arange(100),np.arange(len(val_y))]:
        labels=original_class[valid][ids];yy=val_y[ids];pp=val_scores[ids]>=.5;got=slice_errors(labels,yy,pp)
        assert set(got)==set(np.unique(labels))
        for k in got:
            mask=labels==k;errors=int(np.sum(yy[mask]!=pp[mask]));assert got[k]['n']==int(mask.sum()) and got[k]['errors']==errors
            assert np.isclose(got[k]['error_rate'],errors/mask.sum())
check('Confusion counts',20,test_counts)
check('Metrics and zero denominators',20,test_metrics)
check('F1 threshold',25,test_f1)
check('Cost threshold and ties',20,test_cost)
check('Original-class slices',15,test_slices)
print(json.dumps(checks,ensure_ascii=False,indent=2))
raw_score=sum(r['points'] for r in checks)
print(json.dumps({'name':(student_first+' '+student_last).strip(),'group':student_group.strip(),'assignment':'MET-09','assignment_id':'MET-09','score':raw_score,'raw_score':raw_score,'max_score':100,'score_kind':'raw'},ensure_ascii=False))

## Ваш отчёт и фотографии / Your report and photographs